# Belief State Visualizations for Generative Processes

Visualizes the **Mixed-State Presentation (MSP)** — the set of all reachable belief states —
for four generative processes from [simplexity](https://github.com/Astera-org/simplexity):

| Process | States | Symbols | Model class |
|---------|--------|---------|-------------|
| **MESS3** | 3 | 3 | HMM |
| **MESS3-2** | 3 | 2 | HMM |
| **RIVER** | 3 | 2 | HMM |
| **FANIZZA** | 4 | 2 | GHMM |

3-state processes are shown on the **2-simplex** (barycentric triangle).  
FANIZZA's 4D belief states are projected to 2D via **PCA** and coloured by P(next obs = 0).

**References:**  
- [Belief-state geometry in LLMs (2507.07432)](https://arxiv.org/pdf/2507.07432)  
- [Geometric theory of hidden Markov models (2405.15943)](https://arxiv.org/abs/2405.15943)

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from simplexity.generative_processes.builder import (
    build_hidden_markov_model,
    build_generalized_hidden_markov_model,
)
from simplexity.generative_processes.hidden_markov_model import HiddenMarkovModel
from simplexity.generative_processes.generalized_hidden_markov_model import GeneralizedHiddenMarkovModel
from simplexity.generative_processes.mixed_state_presentation import (
    MixedStateTreeGenerator,
    LogMixedStateTreeGenerator,
)

print(f"JAX backend: {jax.default_backend()}")

## Helper functions

In [ ]:
# ── Coordinate transform ──────────────────────────────────────────────────────

def barycentric_coords(probs: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Map (N, 3) probability vectors to (x, y) on the 2-simplex.

    Vertices: S₀ = (0, 0),  S₁ = (1, 0),  S₂ = (0.5, √3/2)
    """
    x = probs[:, 1] + 0.5 * probs[:, 2]
    y = (np.sqrt(3) / 2) * probs[:, 2]
    return x, y


# ── Colour encoding ───────────────────────────────────────────────────────────

def belief_to_hex(belief_states: np.ndarray) -> list[str]:
    """Encode 3-state belief vectors as #RRGGBB colours (p0→R, p1→G, p2→B)."""
    rgb = np.clip(belief_states[:, :3], 0.0, 1.0)
    return [
        f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"
        for r, g, b in rgb
    ]


# ── Simplex frame ─────────────────────────────────────────────────────────────

def simplex_frame(
    labels: tuple[str, str, str] = ("S₀", "S₁", "S₂"),
    label_positions: tuple[str, str, str] = ("bottom left", "bottom right", "top center"),
) -> list:
    """Return plotly traces that draw the simplex triangle and label its vertices."""
    sqrt3 = np.sqrt(3)
    V = [(0.0, 0.0), (1.0, 0.0), (0.5, sqrt3 / 2)]
    xs = [v[0] for v in V] + [V[0][0]]
    ys = [v[1] for v in V] + [V[0][1]]
    traces = [
        go.Scatter(
            x=xs, y=ys, mode="lines",
            line=dict(color="#444444", width=1.5),
            showlegend=False, hoverinfo="skip",
        )
    ]
    for (vx, vy), label, pos in zip(V, labels, label_positions):
        traces.append(go.Scatter(
            x=[vx], y=[vy], mode="markers+text",
            marker=dict(color="black", size=9),
            text=[label], textposition=pos, textfont=dict(size=14),
            showlegend=False, hoverinfo="skip",
        ))
    return traces


# ── PCA projection (no sklearn dependency) ────────────────────────────────────

def pca_2d(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Project (N, D) array to 2D via PCA.

    Returns
    -------
    X_2d : (N, 2) projected coordinates
    var_ratios : (2,) explained variance ratios for PC1, PC2
    """
    Xc = X - X.mean(axis=0)
    _, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    X_2d = Xc @ Vt[:2].T
    var_ratios = S[:2] ** 2 / np.sum(S ** 2)
    return X_2d, var_ratios


# ── MSP extraction ────────────────────────────────────────────────────────────

def extract_hmm_msp(
    hmm: HiddenMarkovModel,
    max_sequence_length: int = 10,
) -> tuple[np.ndarray, np.ndarray]:
    """Generate the full MSP for an HMM using log-space arithmetic.

    Returns
    -------
    belief_states : (N, num_states) probability vectors
    probabilities : (N,) sequence probabilities
    """
    gen = LogMixedStateTreeGenerator(hmm, max_sequence_length=max_sequence_length)
    tree = gen.generate()
    values = list(tree.nodes.values())
    log_bs = jnp.array([v.log_belief_state for v in values])
    log_probs = jnp.array([v.log_probability for v in values])
    return np.array(jnp.exp(log_bs)), np.exp(np.array(log_probs))


def extract_ghmm_msp(
    ghmm: GeneralizedHiddenMarkovModel,
    max_sequence_length: int = 10,
    prob_threshold: float = 1e-9,
) -> tuple[np.ndarray, np.ndarray]:
    """Generate the full MSP for a GHMM.

    Returns
    -------
    belief_states : (N, num_states) normalised GHMM belief vectors
    probabilities : (N,) sequence probabilities
    """
    gen = MixedStateTreeGenerator(
        ghmm,
        max_sequence_length=max_sequence_length,
        prob_threshold=prob_threshold,
    )
    tree = gen.generate()
    values = list(tree.nodes.values())
    bs = jnp.array([v.belief_state for v in values])
    probs = jnp.array([v.probability for v in values])
    return np.array(bs), np.array(probs)


# ── Simplex figure builder ────────────────────────────────────────────────────

def plot_simplex_msp(
    bs: np.ndarray,
    probs: np.ndarray,
    title: str,
    vertex_labels: tuple[str, str, str] = ("S₀", "S₁", "S₂"),
    marker_size: int = 4,
) -> go.Figure:
    """Interactive plotly figure of belief states on the 2-simplex.

    Colours: p0 → R, p1 → G, p2 → B.
    """
    x, y = barycentric_coords(bs)
    colours = belief_to_hex(bs)
    hover = [
        (
            f"P({vertex_labels[0]})={bs[i,0]:.4f}<br>"
            f"P({vertex_labels[1]})={bs[i,1]:.4f}<br>"
            f"P({vertex_labels[2]})={bs[i,2]:.4f}<br>"
            f"π = {probs[i]:.2e}"
        )
        for i in range(len(bs))
    ]

    fig = go.Figure()
    for trace in simplex_frame(vertex_labels):
        fig.add_trace(trace)
    fig.add_trace(go.Scatter(
        x=x, y=y, mode="markers",
        marker=dict(color=colours, size=marker_size, opacity=0.8),
        hovertext=hover, hoverinfo="text",
        name=f"Belief states  (n = {len(bs):,})",
    ))

    sqrt3 = np.sqrt(3)
    fig.update_layout(
        title=dict(text=title, font=dict(size=14)),
        xaxis=dict(range=[-0.09, 1.09], visible=False, scaleanchor="y"),
        yaxis=dict(range=[-0.12, sqrt3 / 2 + 0.12], visible=False),
        height=560, width=560,
        plot_bgcolor="white",
        legend=dict(x=0.01, y=0.99, font=dict(size=11)),
    )
    return fig


print("Helpers loaded.")

---
## MESS3

A 3-state, 3-symbol HMM with $\mathbb{Z}_3$ symmetry.  
Parameters control the "stickiness" (`a`) and transition spread (`x`) across states.

The MSP is a fractal subset of the 2-simplex with three-fold rotational symmetry.
At high `a`, belief states cluster near the vertices (strong state memory).

In [ ]:
X_M3, A_M3 = 0.15, 0.6
MAX_SEQ_M3 = 8   # 3 symbols → (3^9 - 1)/2 ≈ 9841 nodes

hmm_mess3 = build_hidden_markov_model("mess3", x=X_M3, a=A_M3)
print(f"MESS3  |  states={hmm_mess3.num_states}  symbols={hmm_mess3.vocab_size}")

bs_m3, probs_m3 = extract_hmm_msp(hmm_mess3, max_sequence_length=MAX_SEQ_M3)
print(f"MSP nodes: {len(bs_m3):,}")

fig_m3 = plot_simplex_msp(
    bs_m3, probs_m3,
    title=f"MESS3  —  MSP on 2-simplex<br>"
          f"<sup>x={X_M3}, a={A_M3}  |  max_seq_len={MAX_SEQ_M3}  |  {len(bs_m3):,} nodes</sup>",
    vertex_labels=("S₀", "S₁", "S₂"),
)
fig_m3.show()

---
## MESS3-2

A **2-symbol** variant of MESS3. Each output is a weighted mixture of two MESS3 emissions,
controlled by `(p, q, r)`.  The hidden state space remains 3-dimensional,
so belief states still live on the 2-simplex, but the geometry changes.

With only 2 symbols the tree grows as $2^{\text{depth}}$, enabling deeper exploration.

In [ ]:
X_M32, A_M32 = 0.15, 0.6
P_M32, Q_M32, R_M32 = 0.7, 0.3, 0.5
MAX_SEQ_M32 = 12   # 2 symbols → 2^12+...+1 = 8191 nodes

hmm_mess32 = build_hidden_markov_model(
    "mess3_2", x=X_M32, a=A_M32, p=P_M32, q=Q_M32, r=R_M32
)
print(f"MESS3-2  |  states={hmm_mess32.num_states}  symbols={hmm_mess32.vocab_size}")

bs_m32, probs_m32 = extract_hmm_msp(hmm_mess32, max_sequence_length=MAX_SEQ_M32)
print(f"MSP nodes: {len(bs_m32):,}")

fig_m32 = plot_simplex_msp(
    bs_m32, probs_m32,
    title=f"MESS3-2  —  MSP on 2-simplex<br>"
          f"<sup>x={X_M32}, a={A_M32}, p={P_M32}, q={Q_M32}, r={R_M32}  |  "
          f"max_seq_len={MAX_SEQ_M32}  |  {len(bs_m32):,} nodes</sup>",
    vertex_labels=("S₀", "S₁", "S₂"),
)
fig_m32.show()

---
## RIVER

A 3-state, 2-symbol HMM with fixed (parameterless) transition structure.  
The asymmetric transitions produce an irregular fractal MSP on the 2-simplex,
unlike the $\mathbb{Z}_3$-symmetric MESS3 structure.

In [ ]:
MAX_SEQ_RV = 14   # 2 symbols → 2^14+...+1 = 32767 nodes; use fewer for speed

hmm_river = build_hidden_markov_model("river")
print(f"RIVER  |  states={hmm_river.num_states}  symbols={hmm_river.vocab_size}")

# River has zeros in its transition matrices; LogMixedStateTreeGenerator
# handles log(0)=-inf correctly via logsumexp, so this is numerically safe.
bs_rv, probs_rv = extract_hmm_msp(hmm_river, max_sequence_length=MAX_SEQ_RV)
print(f"MSP nodes: {len(bs_rv):,}")

fig_rv = plot_simplex_msp(
    bs_rv, probs_rv,
    title=f"RIVER  —  MSP on 2-simplex<br>"
          f"<sup>max_seq_len={MAX_SEQ_RV}  |  {len(bs_rv):,} nodes</sup>",
    vertex_labels=("S₀", "S₁", "S₂"),
    marker_size=3,
)
fig_rv.show()

---
## FANIZZA (GHMM)

A **Generalized** HMM with 4 hidden states and 2 symbols.  
Belief states are **not** probability vectors — they live in an affine subspace
normalised by the principal eigenvector of the state-transition operator.

When `α` is an irrational multiple of `2π` and `λ < 1`,
the MSP traces out a **Cantor set** (fractal of measure zero).
Here `α = 2000` rad, `λ = 0.49`.

**Visualisation:** 2D PCA projection coloured by P(next obs = 0).

In [ ]:
ALPHA_FZ, LAMB_FZ = 2000.0, 0.49
MAX_SEQ_FZ = 12   # 2 symbols → 8191 nodes

ghmm_fanizza = build_generalized_hidden_markov_model("fanizza", alpha=ALPHA_FZ, lamb=LAMB_FZ)
print(f"FANIZZA  |  states={ghmm_fanizza.num_states}  symbols={ghmm_fanizza.vocab_size}")

bs_fz, probs_fz = extract_ghmm_msp(ghmm_fanizza, max_sequence_length=MAX_SEQ_FZ)
print(f"MSP nodes: {len(bs_fz):,}")

# Compute P(next obs = 0) for each belief state (used for colouring)
obs_probs_fz = np.array(jnp.stack([
    ghmm_fanizza.observation_probability_distribution(jnp.array(bs_fz[i]))
    for i in range(len(bs_fz))
]))
p_obs0 = obs_probs_fz[:, 0]   # P(obs = 0)

# PCA: 4D → 2D
bs_2d, var_ratios = pca_2d(bs_fz)
print(f"PCA explained variance: PC1={var_ratios[0]:.1%}, PC2={var_ratios[1]:.1%}")

hover_fz = [
    (
        f"P(obs=0)={p_obs0[i]:.4f}<br>"
        f"π = {probs_fz[i]:.2e}<br>"
        f"b = [{', '.join(f'{v:.3f}' for v in bs_fz[i])}]"
    )
    for i in range(len(bs_fz))
]

fig_fz = go.Figure(go.Scatter(
    x=bs_2d[:, 0], y=bs_2d[:, 1],
    mode="markers",
    marker=dict(
        color=p_obs0,
        colorscale="RdBu",
        reversescale=True,
        size=3,
        opacity=0.8,
        colorbar=dict(title="P(obs=0)", thickness=14, len=0.7),
    ),
    hovertext=hover_fz, hoverinfo="text",
    name=f"Belief states  (n = {len(bs_fz):,})",
))
fig_fz.update_layout(
    title=(
        f"FANIZZA GHMM  —  MSP (2D PCA projection)<br>"
        f"<sup>α={ALPHA_FZ}, λ={LAMB_FZ}  |  "
        f"max_seq_len={MAX_SEQ_FZ}  |  {len(bs_fz):,} nodes  |  "
        f"PC1={var_ratios[0]:.1%}, PC2={var_ratios[1]:.1%}</sup>"
    ),
    xaxis_title=f"PC1  ({var_ratios[0]:.1%})",
    yaxis_title=f"PC2  ({var_ratios[1]:.1%})",
    height=580, width=660,
    plot_bgcolor="white",
    legend=dict(x=0.01, y=0.99),
)
fig_fz.show()

---
## FANIZZA — Observation probability (1D Cantor set)

The Fanizza process with irrational `α` has the property that the observation
probability P(next=0) takes values on a **Cantor set** — a fractal of measure zero
in `[0, 1]`. This 1D projection makes the self-similar structure explicit.

In [ ]:
# Sort by P(obs=0) to reveal gaps characteristic of a Cantor set
sorted_idx = np.argsort(p_obs0)
p_sorted = p_obs0[sorted_idx]
pi_sorted = probs_fz[sorted_idx]

fig_cantor = go.Figure()

# Rug plot: each belief state as a tick on [0,1]
fig_cantor.add_trace(go.Scatter(
    x=p_sorted,
    y=np.zeros_like(p_sorted),
    mode="markers",
    marker=dict(
        color=p_sorted,
        colorscale="RdBu",
        reversescale=True,
        size=3,
        symbol="line-ns",
        line=dict(width=1.5, color=p_sorted, colorscale="RdBu"),
    ),
    hovertext=[f"P(obs=0)={p:.5f}" for p in p_sorted],
    hoverinfo="text",
    showlegend=False,
))

fig_cantor.update_layout(
    title=(
        f"FANIZZA  —  P(next=0) Cantor set structure<br>"
        f"<sup>α={ALPHA_FZ}, λ={LAMB_FZ}  |  {len(bs_fz):,} nodes</sup>"
    ),
    xaxis=dict(title="P(obs = 0)", range=[-0.02, 1.02]),
    yaxis=dict(visible=False, range=[-0.5, 0.5]),
    height=260, width=760,
    plot_bgcolor="white",
)
fig_cantor.show()

---
## Combined dashboard (2 × 2)

Side-by-side comparison of all four processes.

In [ ]:
sqrt3 = np.sqrt(3)

fig_all = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f"MESS3  (x={X_M3}, a={A_M3})",
        f"MESS3-2  (x={X_M32}, a={A_M32}, p={P_M32}, q={Q_M32}, r={R_M32})",
        "RIVER",
        f"FANIZZA  (α={ALPHA_FZ}, λ={LAMB_FZ})  — PCA",
    ],
    horizontal_spacing=0.06,
    vertical_spacing=0.10,
)

# ── Helper to add traces to a subplot cell (adjusts axis references) ─────────
def add_simplex_to_subplot(fig, bs, probs, row, col, vertex_labels=("S₀","S₁","S₂")):
    x, y = barycentric_coords(bs)
    colours = belief_to_hex(bs)
    for trace in simplex_frame(vertex_labels):
        fig.add_trace(trace, row=row, col=col)
    fig.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(color=colours, size=3, opacity=0.8),
            showlegend=False, hoverinfo="skip",
        ),
        row=row, col=col,
    )

# Row 1: MESS3 | MESS3-2
add_simplex_to_subplot(fig_all, bs_m3,  probs_m3,  row=1, col=1)
add_simplex_to_subplot(fig_all, bs_m32, probs_m32, row=1, col=2)

# Row 2, col 1: RIVER
add_simplex_to_subplot(fig_all, bs_rv, probs_rv, row=2, col=1)

# Row 2, col 2: FANIZZA (PCA)
fig_all.add_trace(
    go.Scatter(
        x=bs_2d[:, 0], y=bs_2d[:, 1],
        mode="markers",
        marker=dict(
            color=p_obs0,
            colorscale="RdBu",
            reversescale=True,
            size=2,
            opacity=0.8,
        ),
        showlegend=False,
        hoverinfo="skip",
    ),
    row=2, col=2,
)

# ── Axis styling ──────────────────────────────────────────────────────────────
simplex_axes = dict(
    visible=False,
    range=[-0.09, 1.09],
    scaleanchor=None,
    constrain="domain",
)
simplex_yaxes = dict(visible=False, range=[-0.12, sqrt3 / 2 + 0.12], constrain="domain")

for idx, (r, c) in enumerate([(1,1),(1,2),(2,1)]):
    axis_n = "" if idx == 0 else str(idx + 1)
    fig_all.update_layout(**{
        f"xaxis{axis_n}": dict(visible=False),
        f"yaxis{axis_n}": dict(visible=False),
    })

fig_all.update_layout(
    title="Belief State Geometry  —  Mixed-State Presentations",
    height=900, width=1050,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig_all.show()

---
## Parameter sensitivity: MESS3 with varying `a`

Higher `a` → stronger self-loops → belief states cluster near simplex vertices.  
Lower `a` → more mixing → belief states spread toward the centroid.

In [ ]:
A_VALUES = [0.4, 0.6, 0.8, 0.95]
X_FIXED = 0.15
MAX_SEQ_SWEEP = 7

fig_sweep = make_subplots(
    rows=1, cols=len(A_VALUES),
    subplot_titles=[f"a = {a}" for a in A_VALUES],
    horizontal_spacing=0.04,
)

for col_idx, a_val in enumerate(A_VALUES):
    hmm_sw = build_hidden_markov_model("mess3", x=X_FIXED, a=a_val)
    bs_sw, probs_sw = extract_hmm_msp(hmm_sw, max_sequence_length=MAX_SEQ_SWEEP)
    add_simplex_to_subplot(fig_sweep, bs_sw, probs_sw, row=1, col=col_idx + 1)
    print(f"a={a_val}: {len(bs_sw):,} nodes")

fig_sweep.update_layout(
    title=f"MESS3  —  varying a  (x={X_FIXED}, max_seq_len={MAX_SEQ_SWEEP})",
    height=400, width=1200,
    plot_bgcolor="white", paper_bgcolor="white",
)
for i in range(1, len(A_VALUES) + 1):
    n = "" if i == 1 else str(i)
    fig_sweep.update_layout(**{f"xaxis{n}": dict(visible=False), f"yaxis{n}": dict(visible=False)})

fig_sweep.show()

---
## Parameter sensitivity: FANIZZA with varying `λ`

Smaller `λ` → faster contraction → Cantor set with wider gaps.  
Larger `λ < 1` → slower contraction → denser fractal.

In [ ]:
LAMB_VALUES = [0.2, 0.4, 0.49, 0.7]
ALPHA_FIXED = 2000.0
MAX_SEQ_FZ_SWEEP = 10

fig_fz_sweep = make_subplots(
    rows=1, cols=len(LAMB_VALUES),
    subplot_titles=[f"λ = {l}" for l in LAMB_VALUES],
    horizontal_spacing=0.06,
)

for col_idx, lamb_val in enumerate(LAMB_VALUES):
    ghmm_sw = build_generalized_hidden_markov_model(
        "fanizza", alpha=ALPHA_FIXED, lamb=lamb_val
    )
    bs_sw_fz, _ = extract_ghmm_msp(ghmm_sw, max_sequence_length=MAX_SEQ_FZ_SWEEP)
    obs_sw = np.array(jnp.stack([
        ghmm_sw.observation_probability_distribution(jnp.array(bs_sw_fz[i]))
        for i in range(len(bs_sw_fz))
    ]))
    p0_sw = obs_sw[:, 0]
    bs_sw_2d, _ = pca_2d(bs_sw_fz)

    fig_fz_sweep.add_trace(
        go.Scatter(
            x=bs_sw_2d[:, 0], y=bs_sw_2d[:, 1],
            mode="markers",
            marker=dict(color=p0_sw, colorscale="RdBu", reversescale=True,
                        size=3, opacity=0.8),
            showlegend=False, hoverinfo="skip",
        ),
        row=1, col=col_idx + 1,
    )
    print(f"λ={lamb_val}: {len(bs_sw_fz):,} nodes")

fig_fz_sweep.update_layout(
    title=f"FANIZZA  —  varying λ  (α={ALPHA_FIXED}, max_seq_len={MAX_SEQ_FZ_SWEEP})  — PCA projection",
    height=380, width=1200,
    plot_bgcolor="white", paper_bgcolor="white",
)
fig_fz_sweep.show()